# JurisHub Thailand - AI Content Factory (Colab Version)
This notebook runs the entire JurisHub OCR and Pipeline process heavily utilizing Colab GPUs.

In [ ]:
# 1. Install Dependencies
!pip install -q torch transformers einops accelerate pdfplumber pdf2image PyMuPDF
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
# 2. Start Background Ollama Engine
import subprocess
import threading
import time

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)
print("Ollama server running in background...")

# Pull Models
!ollama pull gemma2:2b
!ollama pull deepseek-r1:8b


In [ ]:
# 3. Upload Project Zip Data
# Zip your ENTIRE law directory (containing ai_pipeline, data, js) and upload it here
from google.colab import files
import zipfile
import os

print("Please upload your 'JurisHub_Project.zip' file...")
uploaded = files.upload()

for filename in uploaded.keys():
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('JurisHub')
    print(f"Extracted {filename} into /content/JurisHub")


In [ ]:
# 4. Run the Pipeline
import os
os.chdir('/content/JurisHub')

!python ai_pipeline/step0_multimodal.py
!python ai_pipeline/step1_classifier.py -m gemma2:2b
!python ai_pipeline/step2_rewriter.py -m deepseek-r1:8b
!python ai_pipeline/step3_extractor.py -m deepseek-r1:8b
!python ai_pipeline/step4_exam_parser.py -m gemma2:2b
!python ai_pipeline/ingest_all.py


In [ ]:
# 5. Export Results back to your PC
import shutil
from google.colab import files

print("Zipping processed data and JS...")
shutil.make_archive('/content/colab_results_database', 'zip', 'data/database')
shutil.make_archive('/content/colab_results_js', 'zip', 'js')

files.download('/content/colab_results_database.zip')
files.download('/content/colab_results_js.zip')
print("Download initiated! Overwrite your local folders with these zips.")
